### 퓨샷 프롬프트

- FewShotPromptTemplate : 복잡한 문맥이나 질문데 대한 답변을 처리할 때 효과적
- FewShotPromptTemplate은 기본적으로 One-Shot, Few-Shot을 포함
- One-Shot : 하나의 답변 예시 제공 / Few-Shot : 두 개 이상의 예시 제공

In [1]:
!pip --version

pip 26.2.1 from D:\hanhwa0902\ex0915\.0915venv\Lib\site-packages\pip (python 3.12)



In [3]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_teddynote import logging
from dotenv import load_dotenv

logging.langsmith("test0915")
load_dotenv()

examples = [
    {
        "question": "스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?",
        "answer": """이 질문에 추가 질문이 필요한가요: 예.
                  추가 질문: 스티브 잡스는 몇 살에 사망했나요?
                  중간 답변: 스티브 잡스는 56세에 사망했습니다.
                  추가 질문: 아인슈타인은 몇 살에 사망했나요?
                  중간 답변: 아인슈타인은 76세에 사망했습니다.
                  최종 답변은: 아인슈타인
                  """,
    },
]

LangSmith 추적을 시작합니다.
[프로젝트명]
test0915


In [4]:
example_prompt = PromptTemplate.from_template(
    "Question:\n{question}\nAnswer:\n{answer}"
)

print(example_prompt.format(**examples[0]))

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
                  추가 질문: 스티브 잡스는 몇 살에 사망했나요?
                  중간 답변: 스티브 잡스는 56세에 사망했습니다.
                  추가 질문: 아인슈타인은 몇 살에 사망했나요?
                  중간 답변: 아인슈타인은 76세에 사망했습니다.
                  최종 답변은: 아인슈타인
                  


In [5]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

question = "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"
final_prompt = prompt.format(question=question)
print(final_prompt)

Question:
스티브 잡스와 아인슈타인 중 누가 더 오래 살았나요?
Answer:
이 질문에 추가 질문이 필요한가요: 예.
                  추가 질문: 스티브 잡스는 몇 살에 사망했나요?
                  중간 답변: 스티브 잡스는 56세에 사망했습니다.
                  추가 질문: 아인슈타인은 몇 살에 사망했나요?
                  중간 답변: 아인슈타인은 76세에 사망했습니다.
                  최종 답변은: 아인슈타인
                  

Question:
Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?
Answer:


In [6]:
from langchain_openai import ChatOpenAI
from langchain_teddynote.messages import stream_response

llm = ChatOpenAI()

answer = llm.stream(final_prompt)
stream_response(answer)

이 질문에 추가 질문이 필요합니다. 추가 질문: Google이 창립된 연도가 몇 년도인가요?

In [7]:
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

chain = prompt | llm | StrOutputParser()

answer = chain.stream(
    {"question": "Google이 창립된 연도에 Bill Gates의 나이는 몇 살인가요?"}
)

stream_response(answer)

이 질문에 추가 질문이 필요한가요: 예.
                  추가 질문: Google이 창립된 연도는 언제인가요?
                  중간 답변: Google이 창립된 연도는 1998년입니다.
                  추가 질문: Bill Gates는 1998년에 몇 살이었나요?
                  중간 답변: Bill Gates는 1998년에 43세였습니다.
                  최종 답변은: 43세